# Inception v1

In [1]:
from utils import load_it_data, evaluation_metrics

it_data_dir = '.'
stimulus_train, stimulus_val, stimulus_test, objects_train, objects_val, objects_test, spikes_train, spikes_val = load_it_data(it_data_dir)

ModuleNotFoundError: No module named 'h5py'

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder

class ObjectsDataset(Dataset):
    def __init__(self, stimuli, objects, device):
        self.stimuli = torch.tensor(stimuli, dtype=torch.float32, device=device)
        self.objects = torch.tensor(objects, dtype=torch.long, device=device)

    def __len__(self):
        return len(self.stimuli)

    def __getitem__(self, idx):
        stimulus = self.stimuli[idx]
        obj = self.objects[idx]
        return stimulus, obj
    
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

label_encoder = LabelEncoder()
objects_train_encoded = label_encoder.fit_transform(objects_train)
objects_val_encoded = label_encoder.fit_transform(objects_val)
num_classes = len(label_encoder.classes_)

train_dataset = ObjectsDataset(stimulus_train, objects_train_encoded, device)
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)

val_inputs = torch.tensor(stimulus_val, dtype=torch.float32, device=device)
val_targets = torch.tensor(objects_val_encoded, dtype=torch.long, device=device)

# Display 1st batch shape
for stimuli_batch, objects_batch in train_loader:
    print(f"Stimuli batch shape: {stimuli_batch.shape}")
    print(f"Spikes batch shape: {objects_batch.shape}")
    print(f"Num classes: {num_classes}")
    break

In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms

# GoogleNet specific transformation
preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

batch_size = 4
classes = tuple(set(objects_train))

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

model = torchvision.models.googlenet(
    num_classes=len(classes),
    weights=torchvision.models.GoogLeNet_Weights.IMAGENET1K_V1
)
print(model)


In [ ]:
correct = 0
total = 0

with torch.no_grad():
    for data in train_loader:
        inputs, labels = data[0].to(device), data[1].to(device)
        outputs = model(inputs)
        _, predicted = torch.max(outputs.logits.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"Accuracy of the network on the {total} test images: {100 * correct / total}%")

In [ ]:
correct_pred = {classname: 0 for classname in classes}
total_pred = {classname: 0 for classname in classes}

with torch.no_grad():
    for data in train_loader:
        inputs, labels = data[0].to(device), data[1].to(device)
        outputs = model(inputs)
        _, predictions = torch.max(outputs, 1)

        for label, prediction in zip(labels, predictions):
            if label == prediction:
                correct_pred[classes[label]] += 1
            total_pred[classes[label]] += 1

for classname, correct_count in correct_pred.items():
    accuracy = 100 * float(correct_count) / total_pred[classname]
    print(f"Accuracy for class: {classname} is {accuracy}%")